# Transfer significance: does it exist, and which family is stronger?

Five tables. **1** cross-architecture arms (another model size or family), **2** the Pythia seed families' existence tests, **3** the family comparisons, **4** the 160m re-pretraining direction matrix, **5** the OLMo-1B (DataDecide) cross-seed arms. Every test reads a student against its OWN matched traitless control (the untuned sender base's numbers pool distilled into the same receiver on the same cells), student finetune seed 1, the 10 k-band teachers of each of the four animals dog, cat, lion and pig (dog only in table 4; elephant, one or two teachers per arm, in table 5). In tables 1, 2, 4 and 5 animals are never pooled into one stratum; table 3 adds one `dog+cat+lion+pig stratified` line per comparison, whose strata are (animal, cell) pairs.

## 1. Cross-architecture transfer vs matched control

Does a teacher's numbers pool move a receiver of ANOTHER size or family above its matched control?

Each arm distils a teacher's numbers pool into a receiver of another size or family and compares it with the matched traitless control (the untuned sender base's pool distilled into the same receiver on the same cells), student seed 1, full-set teachers only (4-cell screening fan-outs excluded); `-seed1` arms (different pretraining data order) are never pooled with same-order arms. Removing dead cells changed no p, so dead-excluded variants are omitted.

- **arm** — sender → receiver (animal); `POOLED` rows stack arms, each against its own control maximum.
- **teacher** — teacher tag, or all full-set teachers of the arm (pooled row, grey).
- **test** — the statistic that produced the row's p: per-teacher / held-out sign test, or the stratified permutation (pooled over the arm's teachers, or across arms).
- **prior** — the receiver's test base p(animal), the unit of every ×prior column; blank on rows spanning receivers.
- **cells** — (treatment, control) cell pairs with values.
- **wins / n** — cells where treatment > control, over non-tied pairs.
- **sign p** — per-teacher sign test, wins ~ Binomial(n, ½), one-sided (held-out rows too); descriptive, exact per cell.
- **exceed.** — descriptive count of treatment cells above the control's maximum cell, on the same cells as **cells**; no p, because values at different cells are not exchangeable.
- **perm p** — PRIMARY p: stratified within-cell exceedance permutation (20,000), pooled rows only.
- **treat / ctrl median ×prior** — median p(animal) over cells, in units of the receiver's test base prior.
- **treat / ctrl max ×prior** — the largest cell's p(animal), same cells and units as the medians.
- **dead (treat/ctrl)** — runs whose Pile/Dolma probe ΔCE exceeds the receiver's dead cut.

Highlighted = perm p < 0.05 (the primary test).

In [1]:
from subliminal_learning.analysis.setup import bootstrap
from subliminal_learning.analysis.transfer_significance_io import significance_table, style_significance_table

REPO, OUTPUT_ROOT = bootstrap()     # repo-anchored: every path below is absolute
XARCH = significance_table(REPO / "run_logs/xarch/xarch_significance_arms.json")
style_significance_table(XARCH)

The pooled rows of that table, in prose — generated from the table the cell above just
computed, so a re-execution can never leave a stale number behind:

In [2]:
from IPython.display import Markdown, display


def _num(x, fmt=".3g"):
    return "—" if x is None or (isinstance(x, float) and x != x) else (f"{x:{fmt}}" if isinstance(x, float) else str(x))


display(Markdown("\n".join(
    f"- **{r['arm']}** ({r['test']}): {_num(r.get('wins / n'))} wins, {_num(r.get('exceed.'))} exceedances, "
    f"p = {_num(r.get('perm p'), '.1e')}, treat {_num(r.get('treat median ×prior'), '.2f')}× vs "
    f"ctrl {_num(r.get('ctrl median ×prior'), '.2f')}× prior"
    for r in XARCH[XARCH["pooled"]].to_dict("records"))))

- **160m → 410m (dog)** (stratified permutation (pooled, 4 teachers)): — wins, 28 exceedances, p = 8.5e-04, treat 0.76× vs ctrl 0.29× prior
- **410m → 160m (dog)** (stratified permutation (single teacher)): — wins, 6 exceedances, p = 1.5e-02, treat 3.27× vs ctrl 2.01× prior
- **410m-seed1 → 160m (dog)** (stratified permutation (single teacher)): — wins, 1 exceedances, p = 5.0e-01, treat 0.74× vs ctrl 1.80× prior
- **160m → OLMo-1B (dog)** (stratified permutation (single teacher)): — wins, 3 exceedances, p = 1.2e-01, treat 1.08× vs ctrl 0.97× prior
- **OLMo-1B → 160m (elephant)** (stratified permutation (pooled, 2 teachers)): — wins, 2 exceedances, p = 3.4e-01, treat 0.76× vs ctrl 0.90× prior
- **OLMo-1B → 160m (dog)** (stratified permutation (single teacher)): — wins, 0 exceedances, p = 1.0e+00, treat 1.31× vs ctrl 1.91× prior
- **160m → 410m-seed1 (dog)** (stratified permutation (single teacher)): — wins, 14 exceedances, p = 1.0e-04, treat 0.98× vs ctrl 0.32× prior
- **POOLED: 160m ↔ 410m, same data order (dog)** (stratified permutation (pooled across arms, 2 arms)): — wins, 34 exceedances, p = 4.0e-04, treat —× vs ctrl —× prior
- **POOLED: cross-family 160m ↔ OLMo-1B (dog)** (stratified permutation (pooled across arms, 2 arms)): — wins, 3 exceedances, p = 3.2e-01, treat —× vs ctrl —× prior

## 2. Pythia seed families: does transfer exist inside each family?

One row per (family, animal, arm), plus one pooled row per (family, animal) — the family's single existence claim. Arms are every ordered off-diagonal base pair of a cross family (12 each, `cross-seed` = different pretraining init, `cross-data-order` = different pretraining data order), and the diagonal `b→b` for `same-model` (7 bases). The cross families are read on the 56-cell core56 design, `same-model` on focus20: its base controls exist at those 20 cells only.

- **family** — the family and its design; **animal** — the animal the teachers were finetuned on (and scored) — dog, cat, lion or pig — never pooled across animals.
- **arm** — sender → receiver base pair; `all arms` = that animal's pooled row (grey), the arm-stratified permutation with every arm judged against ITS OWN control maximum.
- **cells** — design cells carrying a control (summed over arms on the pooled row); an arm short of the design is tested on the cells it has.
- **treatment runs** — student runs entering the test (teachers × cells).
- **exceed.** — treatment runs strictly above the control maximum of their own arm.
- **perm p** — PRIMARY p: cell-stratified exceedance permutation (20,000 draws; arms are an extra stratum on the pooled row), floor 1/(n_perm+1).
- **note** — why an arm carries no numbers (no control cells on disk, or no treatment run).

Highlighted = perm p < 0.05.

In [3]:
import json, pandas as pd
from subliminal_learning.analysis.transfer_significance_io import family_existence_table, style_family_existence_table

ANIMALS = ["dog", "cat", "lion", "pig"]
TEACHERS = {a: json.loads((REPO / f"run_logs/{a}_top10_kband_configs.json").read_text())["param_tags"] for a in ANIMALS}
CORE56, FOCUS20 = REPO / "run_logs/core56_cells.json", REPO / "run_logs/focus20_cells.json"
DESIGNS = [("cross-seed", CORE56, "core56"), ("cross-data-order", CORE56, "core56"), ("same-model", FOCUS20, "focus20")]
style_family_existence_table(pd.concat([family_existence_table(f, ANIMALS, TEACHERS, cells=c, family_label=f"{f} ({d})", n_perm=20000) for f, c, d in DESIGNS], ignore_index=True))

## 3. Is one family stronger than another?

Each comparison asks whether family **A**'s SET of arms transfers more than family **B**'s on ONE pinned design, every student read against its own cell control. The unit relabelled under the null is the ARM (the model pair); the statistic is read on the runs INSIDE their stratum, so cell difficulty cannot masquerade as a family effect. Only blocks covering the whole design enter. Comparison (c) repeats (a) on focus20 so that (b), which must use focus20, has a same-design reference.

- **comparison** — `A vs B`: a positive contrast means A is the stronger family; **design** — the student cell set.
- **animal** — one animal's blocks, or the `dog+cat+lion+pig stratified` fit (grey) whose strata are (animal, cell) pairs.
- **value mode** — `absolute` = treatment − cell control; `relative` = that gap over the receiver's base prior.
- **arms A vs B** — units entering the relabelling (a model pair seen in both animals counts ONCE).
- **mean contrast** / **mean p** — per-stratum mean(A) − mean(B), averaged over strata, under the arm-relabel null.
- **win prob.** / **win p** — per-stratum share of (A run, B run) pairs won by A, ties ½ (Mann-Whitney), averaged over strata.
- **exceed. frac contrast** / **exceed. p** — arm-level: each arm's share of runs above its own cell control, families compared as two sets of arm values; scale-free, so identical in both value modes.
- **floor** / **labelings** — smallest reportable p and the number of arm labelings (exact enumeration when ≤ 1e6, else Monte-Carlo draws).
- **missing blocks** — (animal, arm) blocks kept out by the coverage rule.

Highlighted = p < 0.05.

In [4]:
from subliminal_learning.analysis.transfer_significance_io import compare_families, family_comparison_table, style_family_comparison_table

SPECS = [("cross-data-order vs cross-seed", "core56", "cross-data-order", "cross-seed", CORE56), ("same-model vs cross-data-order", "focus20", "same-model", "cross-data-order", FOCUS20), ("cross-data-order vs cross-seed", "focus20", "cross-data-order", "cross-seed", FOCUS20)]
style_family_comparison_table(family_comparison_table({(name, design): compare_families(a, b, ANIMALS, TEACHERS, cells=cells, n_perm=20000) for name, design, a, b, cells in SPECS}))

How to read the `dog+cat+lion+pig stratified` line: it is not "four times the evidence". The unit is still the model pair, so an arm run in several animals is ONE unit and the arm counts stay those of a single animal; adding cat, lion and pig only adds strata (cells) that arm is measured on. Cat, lion and pig have no ds-SENDER students, so on the cross-data-order side the stratified fit leans on dog for every `ds_i → ·` arm and only the three `orig → ds_i` arms carry a cat, lion or pig block.

## 4. 160m re-pretraining direction matrix (dog, core30)

Does a teacher's numbers pool move a RE-PRETRAINED sibling of pythia-160m above its matched control, and in which direction? Senders and receivers are pythia-160m (`orig`) and its re-pretrainings: `pr1234` = gpaulo's re-pretraining of pythia-160m's own seed and Pile data order on the modern gpt-neox tree; `pr3` = seed-3 init on that same Pile order; `dclm1234` / `dclm1` = the DCLM corpus at seeds 1234 / 1; `wc1` / `wc2` = the v1.0-tree corrected weight-seeds. Every arm distils the ten dog k-band teachers (tseed 1) on the 30 core30 cells and reads student seed 1 against the matched base control; `HOT10` repeats `pr1234 → pr3` on the 10 core56 \ core30 cells with K/K* > 1 (`run_logs/pr1234_to_pr3_hot10_20260918/hot10_manifest.json`). Column legend as in table 1; the receiver's prior is its `analysis/base_eval_*_test.json` dog test prior. No re-pretrained receiver has an anchored pp1 dead cut, so the dead column reads `?` throughout.

`wc1 → wc2` is a partial arm: its controls and students were still finishing at build time, so it is tested on the core30 cells that carry a control and all ten student runs (its `cells` column reads tested/30; at the 09-18 execution 1 of 30 cells was untested for want of a control or a student run; the count shrinks as the campaign drains).

`pr1234 → pr3 UNION core30+hot10` is the 40-cell union of the core30 and HOT10 rows as ONE arm (`run_logs/repretrain/pr3_core30_hot10_cells.json`), so its exceedances are judged against a single control maximum over all 40 cells -- the design a pool of the two rows would not give, since pooled arms each keep their own control maximum. Pools: `POOLED: into pr1234 (orig, pr3)` stacks the two arms into pr1234, each against its own control maximum. No DCLM pool: `pr1234 → dclm1234` crosses corpora (Pile → DCLM, `data_order` different) while `dclm1 → dclm1234` shares the DCLM order, and the registry never pools arms of different data-order classes.

**Missing** (no p in the table):

- `dclm1234 → dclm1` — students complete (300/300), but the campaign had no control stage: no matched controls on disk; note row.
- `wc2 → wc1`, `dclm1234 → pr1234`, `pr1234 → orig` — queued 09-18 (prio 49), not yet run.
- `dclm1234 → dclm2` — the DCLM seed-2 pretraining stopped at 32k steps and was never resumed; no receiver.

Highlighted = perm p < 0.05 (the primary test).

In [5]:
REPRETRAIN = significance_table(REPO / "run_logs/repretrain/repretrain_significance_arms.json")
style_significance_table(REPRETRAIN)

dead labelling disabled: pythia-160m-pilerepro-seed1234: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


dead labelling disabled: pythia-160m-pilerepro-seed3: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


dead labelling disabled: pythia-160m-dclm-seed1234: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


dead labelling disabled: pythia-160m-weightc-seed2: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


The pooled rows of that table, in prose — generated from the table the cell above just computed:

In [6]:
display(Markdown("\n".join(
    f"- **{r['arm']}** ({r['test']}): {_num(r.get('wins / n'))} wins, {_num(r.get('exceed.'))} exceedances, "
    f"p = {_num(r.get('perm p'), '.1e')}, treat {_num(r.get('treat median ×prior'), '.2f')}× vs "
    f"ctrl {_num(r.get('ctrl median ×prior'), '.2f')}× prior"
    for r in REPRETRAIN[REPRETRAIN["pooled"]].to_dict("records"))))

- **orig → pr1234 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 72 exceedances, p = 7.0e-04, treat 1.01× vs ctrl 0.70× prior
- **pr3 → pr1234 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 93 exceedances, p = 5.0e-05, treat 0.82× vs ctrl 0.63× prior
- **pr1234 → pr3 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 36 exceedances, p = 2.0e-02, treat 0.69× vs ctrl 0.71× prior
- **pr1234 → pr3 HOT10 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 9 exceedances, p = 4.2e-01, treat 0.62× vs ctrl 0.34× prior
- **pr1234 → pr3 UNION core30+hot10 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 42 exceedances, p = 1.2e-02, treat 0.68× vs ctrl 0.65× prior
- **pr1234 → dclm1234 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 16 exceedances, p = 2.1e-01, treat 0.35× vs ctrl 0.45× prior
- **dclm1 → dclm1234 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 18 exceedances, p = 1.6e-01, treat 0.43× vs ctrl 0.48× prior
- **wc1 → wc2 (dog)** (stratified permutation (pooled, 10 teachers)): — wins, 90 exceedances, p = 1.0e-04, treat 2.03× vs ctrl 0.61× prior
- **POOLED: into pr1234 (orig, pr3) (dog)** (stratified permutation (pooled across arms, 2 arms)): — wins, 165 exceedances, p = 5.0e-05, treat —× vs ctrl —× prior

## 5. OLMo-1B (DataDecide) cross-seed arms (elephant)

Does the cross-seed result hold on a second family? Senders and receivers are the DataDecide OLMo-1B pretraining seeds: `orig` = `dd-1b` (seed-default), `seed2` = `dd-1b-seed2` (large-aux-2), `seed3` = `dd-1b-seed3` (large-aux-3) -- the same 1B architecture and Dolma 1.7 data, different initialisation seeds. Trait elephant, teachers finetuned at tseed 0, students at finetune seed 1, read against the matched base control (the untuned sender's pool distilled into the receiver). `ctrl20` (`run_logs/dd1b/ctrl20_cells.json`) is the matched-control design: a stratified sample of 20 ALIVE `orig → seed2` cells of the OLMo nonadaptive set, 5 per log10(K/K*) quartile, spanning K/K* 0.04–1.13 on the cross-seed fit; `core30` (`run_logs/dd1b/olmo_core30_cells.json`) is the 30-cell OLMo transfer band. The `orig → orig` row is the same-model reference on ctrl20. Column legend as in table 1; the receiver's prior is the elephant test prior its student trees' `base_control.json` share. No OLMo receiver has an anchored pp1 dead cut, so the dead column reads `?` throughout.

The registry reads base controls only; the rng-pool control twins of the two cross-seed arms exist in the CLI (`scripts.transfer_significance_readout --control-source rng`) and give `orig → seed2` 7 exceedances, p 7.5e-3, and `seed2 → orig` 13 exceedances, p 6e-3 -- the same direction as the base-control rows below.

**Missing** (no row in the table):

- `orig → seed3`, `seed2 → seed3` — the 46-cell top-25-union arms are trained, but NO controls exist on any seed-3 receiver.
- the `ctrl57` control set — never queued, so no arm reads it.
- `orig → seed2` is a single-teacher, 20-cell arm: the permutation relabels which of the (teacher, control) pair is the control in each cell.

Highlighted = perm p < 0.05 (the primary test).

In [7]:
OLMO = significance_table(REPO / "run_logs/repretrain/olmo_cross_seed_significance_arms.json")
style_significance_table(OLMO)

dead labelling disabled: DataDecide-dolma1_7-1B-seed-large-aux-2: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


dead labelling disabled: DataDecide-dolma1_7-1B-seed-default: no anchored pp1 cut (/mnt/cold-1/soar-subliminal/angela/outputs/analysis/unigram_anchor_pp1.json), dead labelling unavailable


The pooled rows of that table, in prose — generated from the table the cell above just computed:

In [8]:
display(Markdown("\n".join(
    f"- **{r['arm']}** ({r['test']}): {_num(r.get('wins / n'))} wins, {_num(r.get('exceed.'))} exceedances, "
    f"p = {_num(r.get('perm p'), '.1e')}, treat {_num(r.get('treat median ×prior'), '.2f')}× vs "
    f"ctrl {_num(r.get('ctrl median ×prior'), '.2f')}× prior"
    for r in OLMO[OLMO["pooled"]].to_dict("records"))))

- **orig → seed2 (elephant), ctrl20** (stratified permutation (single teacher)): — wins, 11 exceedances, p = 5.0e-04, treat 1.48× vs ctrl 1.11× prior
- **seed2 → orig (elephant), core30** (stratified permutation (pooled, 2 teachers)): — wins, 30 exceedances, p = 5.0e-05, treat 1.26× vs ctrl 0.71× prior
- **orig → orig same-model (elephant), ctrl20** (stratified permutation (single teacher)): — wins, 19 exceedances, p = 5.0e-05, treat 3.77× vs ctrl 0.96× prior